# Train NEXUS Command Classifiers (Tier 3 — Skip ASR for Known Commands)

This notebook trains **multiple** openWakeWord command classifiers in a single Colab session.
Each model detects a spoken command like "open youtube" or "open gmail" directly from audio —
**no ASR needed**. When a command model fires, NEXUS executes the action in ~200ms instead of ~30s.

**Based on**: `train_nexus_oww.ipynb` (the wake-word trainer). This notebook reuses the same
infrastructure (Piper TTS, FMA noise, ACAV100M negatives, MIT RIRs) but trains a **loop** of commands.

**Runtime**: ~4-6 hours on Colab Pro (L4 GPU + High RAM) for 10 commands.
  - Setup (install + downloads): ~30 min (done ONCE)
  - Per command (generate clips + augment + train + export): ~15-25 min

**Output**: Multiple `.onnx` files (one per command), each ~800KB.

## Commands to train
Edit the `COMMANDS` list in the config cell to choose which commands to train.
Each command needs:
  - `phrase`: what the user says (e.g. "open youtube")
  - `model_name`: filename for the ONNX model (e.g. "open_youtube")
  - `negatives`: similar-sounding phrases to exclude (reduces false positives)
  - `intent`: the structured intent to emit when detected

## Instructions
1. **Runtime → Change runtime type → T4 GPU** (or L4 GPU + High RAM on Colab Pro)
2. **Runtime → Run all**
3. Each model is auto-downloaded as it completes. Place all `.onnx` files in:
   `src-tauri/resources/oww/commands/`

## 1. Install deps (same as wake-word notebook)

In [ ]:
# Native deps
!apt-get install -y -qq cmake espeak-ng espeak-ng-data libespeak-ng-dev libsndfile1 pkg-config build-essential ffmpeg unzip 2>&1 | tail -3

# Python deps — same order as wake-word notebook
!pip install -q piper-phonemize-cross
!pip install -q \
    webrtcvad \
    mutagen==1.47.0 \
    torchinfo \
    torchmetrics \
    pyyaml \
    tqdm \
    datasets \
    soundfile \
    audiomentations \
    torch_audiomentations \
    pronouncing \
    onnxruntime \
    onnx \
    speechbrain \
    acoustics \
    scipy \
    requests \
    huggingface_hub
!pip install -q --no-deps piper-tts

import sys
print('Python:', sys.version)
import torch, torchinfo, torchmetrics, scipy, numpy
print(f'  torch: {torch.__version__}  cuda: {torch.cuda.is_available()}')
from tqdm import tqdm
import yaml, mutagen, pronouncing
import torchaudio, audiomentations, torch_audiomentations
import speechbrain, acoustics
import onnx, onnxruntime, soundfile, requests
from piper_phonemize import phonemize_espeak
from piper import PiperVoice, SynthesisConfig
print('  All deps import cleanly.')

## 1b. Mount Google Drive (for checkpointing — survives session disconnects)

Colab free tier has a **12-hour session limit** and a **~90 minute idle timeout**. If the session disconnects, all files in `/content` are lost. We mount Google Drive so that:
- Each trained `.onnx` model is saved to Drive as it completes
- If the session crashes, you can re-run the notebook and it will **skip already-trained models** (resume from where you left off)
- The `command_intents.json` is also saved to Drive

**You will be prompted to authorize Google Drive access. Click the link and paste the code.**

In [ ]:
import os, shutil

# Mount Google Drive for checkpointing
DRIVE_MOUNTED = False
DRIVE_MODELS_DIR = '/content/drive/MyDrive/nexus_command_models'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
    # Check for existing models from previous runs
    existing = [f for f in os.listdir(DRIVE_MODELS_DIR) if f.endswith('.onnx')]
    if existing:
        print(f'  Found {len(existing)} existing models in Drive:')
        for f in sorted(existing):
            print(f'    {f} ({os.path.getsize(os.path.join(DRIVE_MODELS_DIR, f))/1e3:.0f} KB)')
        print('  → These will be skipped during training (resume capability)')
    else:
        print('  No existing models in Drive — starting fresh')
    DRIVE_MOUNTED = True
    print(f'  OK Google Drive mounted: {DRIVE_MODELS_DIR}')
except Exception as e:
    print(f'  WARNING: Could not mount Google Drive: {e}')
    print('  Models will only be saved to /content (ephemeral — lost on disconnect)')
    print('  To enable checkpointing, re-run this cell and authorize Drive access')
    DRIVE_MOUNTED = False

# Keep-alive helper: call this during long training to prevent idle timeout
# Colab disconnects after ~90 min of no interaction. This prints a heartbeat
# which counts as "interaction" in the Colab UI.
import time
_last_heartbeat = time.time()
def heartbeat(label=''):
    global _last_heartbeat
    now = time.time()
    elapsed = now - _last_heartbeat
    if elapsed > 60:  # Print at most once per minute
        print(f'  [heartbeat] {time.strftime("%H:%M:%S")} — {label} (alive, {elapsed:.0f}s since last)', flush=True)
        _last_heartbeat = now

In [ ]:
import os, sys
os.chdir('/content')

# piper-sample-generator (pinned to flat-layout commit)
PSG_DIR = '/content/piper-sample-generator'
PSG_PIN = '1a8c49bd29b3a132721086ee88f2253f788594a8^'
PSG_GS  = f'{PSG_DIR}/generate_samples.py'
if not os.path.exists(PSG_GS):
    !rm -rf {PSG_DIR}
    !git clone -q https://github.com/rhasspy/piper-sample-generator {PSG_DIR}
!cd {PSG_DIR} && git fetch -q --all && git checkout -q {PSG_PIN}
assert os.path.exists(PSG_GS), 'piper-sample-generator pin failed'
print(f'  OK piper-sample-generator')

# libritts model (~200 MB)
PIPER_MODEL = f'{PSG_DIR}/models/en_US-libritts_r-medium.pt'
PIPER_MODEL_URL = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
if not os.path.exists(PIPER_MODEL) or os.path.getsize(PIPER_MODEL) < 100_000_000:
    !mkdir -p {PSG_DIR}/models
    !wget -q --tries=5 --timeout=300 -O {PIPER_MODEL} {PIPER_MODEL_URL}
assert os.path.getsize(PIPER_MODEL) > 100_000_000
print(f'  OK libritts model: {os.path.getsize(PIPER_MODEL)/1e6:.0f} MB')

# openwakeword
OWW_DIR = '/content/openwakeword'
OWW_TRAIN = f'{OWW_DIR}/openwakeword/train.py'
if not os.path.exists(OWW_TRAIN):
    !rm -rf {OWW_DIR}
    !git clone -q https://github.com/dscripka/openwakeword {OWW_DIR}
    !pip install -q -e {OWW_DIR}
assert os.path.exists(OWW_TRAIN)
if OWW_DIR not in sys.path:
    sys.path.insert(0, OWW_DIR)
for _m in list(sys.modules):
    if _m.startswith('openwakeword'):
        del sys.modules[_m]
import openwakeword
assert openwakeword.__file__ is not None
print(f'  OK openwakeword: {openwakeword.__file__}')

## 3. Apply runtime patches (same 6 patches as wake-word notebook)

In [ ]:
import os, glob

# Patch A: torchaudio.set_audio_backend → pass
for path in glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py'):
    !sed -i 's|torchaudio.set_audio_backend("soundfile")|pass  # patched|' "{path}"

# Patch B: copy generate_samples.py
src = '/content/piper-sample-generator/generate_samples.py'
dst = '/content/openwakeword/openwakeword/generate_samples.py'
if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
    !cp "{src}" "{dst}"

# Patch C: HF Hub timeouts
os.environ['HF_HUB_ETAG_TIMEOUT'] = '120'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
import huggingface_hub.constants as hfc
for attr in ['DEFAULT_ETAG_TIMEOUT', 'DEFAULT_DOWNLOAD_TIMEOUT',
             'HF_HUB_ETAG_TIMEOUT', 'HF_HUB_DOWNLOAD_TIMEOUT']:
    if hasattr(hfc, attr): setattr(hfc, attr, 120)

# Patch D: torchaudio.info shim
import torchaudio
init_path = torchaudio.__file__
SHIM_MARKER = '# PATCH: info() shim for torchaudio 2.x'
with open(init_path) as f:
    content = f.read()
if SHIM_MARKER not in content:
    shim = (f'\n\n{SHIM_MARKER}\n'
            'def info(file_path, *args, **kwargs):\n'
            '    import soundfile as _sf\n'
            '    si = _sf.info(str(file_path))\n'
            '    return type("_TorchaudioInfo", (), {\n'
            '        "num_frames": si.frames, "sample_rate": si.samplerate,\n'
            '        "num_channels": si.channels, "bits_per_sample": 16,\n'
            '        "encoding": "PCM_S",\n'
            '    })()\n')
    with open(init_path, 'a') as f: f.write(shim)
    import importlib; importlib.reload(torchaudio)

# Patch E: generate_samples model arg default
TARGET = '/content/piper-sample-generator/generate_samples.py'
!sed -i 's|model: Union\[str, Path\],|model: Union[str, Path] = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",|' "{TARGET}"
!cp "{TARGET}" /content/openwakeword/openwakeword/generate_samples.py

# Patch F: train.py val dtype cast
TRAIN_PY = '/content/openwakeword/openwakeword/train.py'
!sed -i 's|val_predictions = self.model(x_val)$|val_predictions = self.model(x_val.float())|' "{TRAIN_PY}"

print('All 6 patches applied.')

## 4. Download shared data (MIT RIRs, FMA, ACAV100M — done ONCE)

In [ ]:
import os, time

# Shared OWW models (melspectrogram + embedding)
OWW_MODELS_DIR = '/content/oww_models'
os.makedirs(OWW_MODELS_DIR, exist_ok=True)
for fname in ['melspectrogram.onnx', 'embedding_model.onnx']:
    path = f'{OWW_MODELS_DIR}/{fname}'
    if not os.path.exists(path) or os.path.getsize(path) < 100_000:
        url = f'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{fname}'
        !wget --tries=5 --timeout=120 -O {path} {url}
    sz = os.path.getsize(path)
    assert sz > 100_000, f'{fname} download failed: only {sz} bytes'
    print(f'  OK {fname}: {sz/1e6:.1f} MB')

# MIT RIRs
RIR_DIR = '/content/mit_rirs'
if not os.path.exists(RIR_DIR) or len(os.listdir(RIR_DIR)) < 250:
    from huggingface_hub import snapshot_download
    for attempt in range(6):
        try:
            snapshot_download('davidscripka/MIT_environmental_impulse_responses',
                              repo_type='dataset', local_dir='/content/mit_rirs_raw')
            break
        except Exception as e:
            print(f'  MIT RIRs retry {attempt+1}: {e}')
            time.sleep(5)
    os.makedirs(RIR_DIR, exist_ok=True)
    import soundfile as sf, glob
    from scipy.signal import resample_poly
    for f in glob.glob('/content/mit_rirs_raw/**/*.wav', recursive=True):
        data, sr = sf.read(f)
        if sr != 16000:
            data = resample_poly(data.astype('float32'), 16000, sr)
        sf.write(f'{RIR_DIR}/{os.path.basename(f)}', data, 16000)
    # Clean up raw download to save ~0.5 GB disk
    import shutil
    shutil.rmtree('/content/mit_rirs_raw', ignore_errors=True)
    print('  Cleaned up mit_rirs_raw (saved ~0.5 GB)')
rir_count = len(os.listdir(RIR_DIR)) if os.path.exists(RIR_DIR) else 0
print(f'  OK MIT RIRs: {rir_count} files')

# FMA small dataset (~8 GB) — with retry + fallback
FMA_ZIP = '/content/fma_small.zip'
FMA_DIR = '/content/fma'
FMA_WAV = '/content/fma_wav'
FMA_OK = False
if not os.path.exists(FMA_DIR) or not os.path.exists(FMA_WAV) or len(os.listdir(FMA_WAV)) < 100:
    # Try downloading FMA small
    for attempt in range(3):
        if not os.path.exists(FMA_ZIP) or os.path.getsize(FMA_ZIP) < 7_000_000_000:
            print(f'  FMA download attempt {attempt+1}/3...')
            !wget --tries=3 --timeout=300 -O {FMA_ZIP} https://os.unil.cloud.switch.ch/fma/fma_small.zip
        if os.path.exists(FMA_ZIP) and os.path.getsize(FMA_ZIP) > 7_000_000_000:
            !unzip -q -o {FMA_ZIP} -d /content/
            # Delete the 8 GB zip after unzip to save disk space
            os.remove(FMA_ZIP)
            print('  Deleted FMA zip after unzip (saved ~8 GB)')
            FMA_OK = True
            break
        else:
            sz = os.path.getsize(FMA_ZIP) if os.path.exists(FMA_ZIP) else 0
            print(f'  FMA download incomplete: {sz/1e9:.2f} GB (need ~8 GB), retrying...')
            if os.path.exists(FMA_ZIP):
                os.remove(FMA_ZIP)
            time.sleep(10)
    if not FMA_OK:
        print('  WARNING: FMA download failed after 3 attempts.')
        print('  Will use ACAV100M + synthetic noise as background audio instead.')
else:
    FMA_OK = True
    # Clean up zip if it exists from a previous partial run
    if os.path.exists(FMA_ZIP):
        os.remove(FMA_ZIP)
print(f'  OK FMA: {FMA_OK}')

# Convert FMA MP3s to WAVs (1500 clips) — only if FMA downloaded
if FMA_OK and (not os.path.exists(FMA_WAV) or len(os.listdir(FMA_WAV)) < 1000):
    os.makedirs(FMA_WAV, exist_ok=True)
    import glob, subprocess
    mp3s = sorted(glob.glob(f'{FMA_DIR}/**/*.mp3', recursive=True))[:1500]
    for mp3 in mp3s:
        wav = f'{FMA_WAV}/{os.path.splitext(os.path.basename(mp3))[0]}.wav'
        if not os.path.exists(wav):
            subprocess.run(['ffmpeg', '-y', '-i', mp3, '-ar', '16000',
                           '-ac', '1', '-t', '30', wav],
                          capture_output=True)
fma_wav_count = len(os.listdir(FMA_WAV)) if os.path.exists(FMA_WAV) else 0
print(f'  OK FMA WAVs: {fma_wav_count} files')

# ACAV100M features (~17 GB) — with validation + fallback
# This is the largest download. On Colab free tier it may fail due to disk
# space or network timeouts. We handle this gracefully.
ACAV_SRC = '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
ACAV_TRAIN = '/content/acav_train_subset.npy'
ACAV_VAL = '/content/acav_val_subset.npy'
ACAV_OK = False

# Check if we already have processed subsets (from a previous run)
if os.path.exists(ACAV_TRAIN) and os.path.exists(ACAV_VAL):
    ACAV_OK = True
    print(f'  OK ACAV subsets already processed (cached from previous run)')
elif os.path.exists(ACAV_SRC) and os.path.getsize(ACAV_SRC) > 1_000_000_000:
    # File exists and is > 1 GB — try to load it
    print(f'  ACAV source file exists: {os.path.getsize(ACAV_SRC)/1e9:.1f} GB')
    ACAV_OK = True
else:
    # Need to download
    # Clean up any partial/empty file first
    if os.path.exists(ACAV_SRC):
        sz = os.path.getsize(ACAV_SRC)
        print(f'  Removing incomplete ACAV file: {sz/1e6:.1f} MB (too small)')
        os.remove(ACAV_SRC)

    print('  Downloading ACAV100M features (~17 GB)... this takes 10-40 min')
    # Use huggingface_hub for better retry/resume support
    from huggingface_hub import hf_hub_download
    for attempt in range(3):
        try:
            result = hf_hub_download(
                repo_id='dscripka/openwakeword_features',
                filename='openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
                repo_type='dataset',
                local_dir='/content/',
            )
            # hf_hub_download saves to a cache; copy/symlink to expected path
            if os.path.exists(result) and os.path.getsize(result) > 1_000_000_000:
                if result != ACAV_SRC:
                    import shutil
                    shutil.move(result, ACAV_SRC)
                ACAV_OK = True
                print(f'  OK ACAV downloaded: {os.path.getsize(ACAV_SRC)/1e9:.1f} GB')
                break
            else:
                print(f'  ACAV download too small, retrying...')
        except Exception as e:
            print(f'  ACAV download attempt {attempt+1} failed: {e}')
            time.sleep(10)

    # Fallback: try wget if hf_hub_download failed
    if not ACAV_OK:
        print('  Trying wget fallback for ACAV...')
        !wget --tries=3 --timeout=600 -c -O {ACAV_SRC} \
            'https://huggingface.co/datasets/dscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
        if os.path.exists(ACAV_SRC) and os.path.getsize(ACAV_SRC) > 1_000_000_000:
            ACAV_OK = True
            print(f'  OK ACAV downloaded via wget: {os.path.getsize(ACAV_SRC)/1e9:.1f} GB')
        else:
            sz = os.path.getsize(ACAV_SRC) if os.path.exists(ACAV_SRC) else 0
            print(f'  WARNING: ACAV download failed: only {sz/1e6:.1f} MB received')

if ACAV_OK and not os.path.exists(ACAV_TRAIN):
    # Process ACAV into train/val subsets
    import numpy as np
    print('  Processing ACAV subsets (this takes a few minutes)...')
    try:
        full = np.load(ACAV_SRC, mmap_mode='r')
        print(f'  ACAV loaded: shape={full.shape}, dtype={full.dtype}')
        train_chunk = full[:len(full)//10]
        np.save(ACAV_TRAIN, train_chunk)
        val_chunk = full[len(full)//10:len(full)//10 + len(full)//100]
        val_flat = val_chunk.reshape(-1, val_chunk.shape[-1])
        np.save(ACAV_VAL, val_flat)
        del full, train_chunk, val_chunk
        # Free disk space — remove the 17 GB source file
        if os.path.exists(ACAV_SRC):
            os.remove(ACAV_SRC)
            print('  Removed 17 GB ACAV source (subsets saved)')
    except Exception as e:
        print(f'  ERROR processing ACAV: {e}')
        ACAV_OK = False
        # Clean up corrupted file
        if os.path.exists(ACAV_SRC):
            os.remove(ACAV_SRC)

if ACAV_OK:
    train_sz = os.path.getsize(ACAV_TRAIN)/1e9 if os.path.exists(ACAV_TRAIN) else 0
    val_sz = os.path.getsize(ACAV_VAL)/1e6 if os.path.exists(ACAV_VAL) else 0
    print(f'  OK ACAV train: {train_sz:.1f} GB')
    print(f'  OK ACAV val: {val_sz:.0f} MB')
else:
    print('  WARNING: ACAV100M not available.')
    print('  Training will use FMA WAVs + synthetic noise as negatives instead.')
    print('  This produces slightly less robust models but still works.')

# Print disk usage summary
import subprocess
result = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print('\n--- Disk usage ---')
print(result.stdout.strip())

print('\n--- Shared data summary ---')
print(f'  OWW models:  melspectrogram + embedding')
print(f'  MIT RIRs:    {rir_count} files')
print(f'  FMA WAVs:    {fma_wav_count} files')
print(f'  ACAV100M:    {"available" if ACAV_OK else "NOT available (using fallback)"}')
print('All shared data ready.')

## 5. Command configuration — EDIT THIS TO CHANGE COMMANDS

Each command trains a separate OWW classifier. The shared infrastructure (Piper TTS, FMA, ACAV, RIRs) is reused across all commands.

Add or remove commands from this list. Each command trains in ~15-25 min.

In [ ]:
# ─── COMMANDS TO TRAIN ───────────────────────────────────────────────
# Each entry trains a separate OWW classifier model.
#
# phrase:      what the user says (used for TTS clip generation)
# model_name:  filename for the .onnx output (no spaces, lowercase)
# negatives:   similar-sounding phrases to exclude (reduces false positives)
# intent:      the structured intent NEXUS will execute when detected
#
# ─── Command Types ───────────────────────────────────────────────────
#
# Type 1 (Fixed):     intent has no needs_param → execute directly, no STT
# Type 2 (Param):     intent has needs_param: true → acoustic trigger fires,
#                      then record 3s of audio + STT to get the parameter
#                      (e.g. song name, search query)
#
# For Type 2, the phrase should be a generic version of the command:
#   "play song in spotify" → detects "play <anything> in spotify"
#   "search on youtube"    → detects "search <anything> on youtube"

COMMANDS = [
    # ═══════════════════════════════════════════════════════════════════
    # CATEGORY A: Fixed Commands (Type 1 — acoustic only, no STT needed)
    # ═══════════════════════════════════════════════════════════════════

    # ── Original 10 (already trained — will be skipped if in Drive) ──
    {
        'phrase': 'open youtube',
        'model_name': 'open_youtube',
        'negatives': ['open you tube', 'open utube', 'open youth tube', 'open u tube'],
        'intent': {'action': 'open_app', 'target': 'youtube'},
    },
    {
        'phrase': 'open gmail',
        'model_name': 'open_gmail',
        'negatives': ['open gamail', 'open gee mail', 'open j mail', 'open email'],
        'intent': {'action': 'open_app', 'target': 'gmail'},
    },
    {
        'phrase': 'open chrome',
        'model_name': 'open_chrome',
        'negatives': ['open crowm', 'open comb', 'open chrome book', 'open krom'],
        'intent': {'action': 'open_app', 'target': 'chrome'},
    },
    {
        'phrase': 'open notepad',
        'model_name': 'open_notepad',
        'negatives': ['open note pad', 'open node pad', 'open no pad', 'open notebook'],
        'intent': {'action': 'open_app', 'target': 'notepad'},
    },
    {
        'phrase': 'open calculator',
        'model_name': 'open_calculator',
        'negatives': ['open calc', 'open calculate', 'open calendar', 'open calculus'],
        'intent': {'action': 'open_app', 'target': 'calculator'},
    },
    {
        'phrase': 'open spotify',
        'model_name': 'open_spotify',
        'negatives': ['open spot ify', 'open spotty fly', 'open spot a fy', 'open spotty'],
        'intent': {'action': 'open_app', 'target': 'spotify'},
    },
    {
        'phrase': 'open discord',
        'model_name': 'open_discord',
        'negatives': ['open dis cord', 'open this cord', 'open disk cord', 'open des cord'],
        'intent': {'action': 'open_app', 'target': 'discord'},
    },
    {
        'phrase': 'open github',
        'model_name': 'open_github',
        'negatives': ['open git hub', 'open get hub', 'open gith ub', 'open git hob'],
        'intent': {'action': 'open_app', 'target': 'github'},
    },
    {
        'phrase': 'open vs code',
        'model_name': 'open_vscode',
        'negatives': ['open vs code', 'open visual studio code', 'open code', 'open v s code'],
        'intent': {'action': 'open_app', 'target': 'vscode'},
    },
    {
        'phrase': 'open figma',
        'model_name': 'open_figma',
        'negatives': ['open fig ma', 'open fig mma', 'open big ma', 'open sigma'],
        'intent': {'action': 'open_app', 'target': 'figma'},
    },

    # ── New: 20 more fixed commands ──────────────────────────────────
    {
        'phrase': 'open slack',
        'model_name': 'open_slack',
        'negatives': ['open stack', 'open slak', 'open smack', 'open slick'],
        'intent': {'action': 'open_app', 'target': 'slack'},
    },
    {
        'phrase': 'open notion',
        'model_name': 'open_notion',
        'negatives': ['open no shun', 'open motion', 'open potion', 'open ocean'],
        'intent': {'action': 'open_app', 'target': 'notion'},
    },
    {
        'phrase': 'open terminal',
        'model_name': 'open_terminal',
        'negatives': ['open term in all', 'open turn a mull', 'open term nal', 'open tur minal'],
        'intent': {'action': 'open_app', 'target': 'terminal'},
    },
    {
        'phrase': 'open explorer',
        'model_name': 'open_explorer',
        'negatives': ['open ex plor er', 'open explore', 'open ex plorer', 'open x plorer'],
        'intent': {'action': 'open_app', 'target': 'explorer'},
    },
    {
        'phrase': 'open settings',
        'model_name': 'open_settings',
        'negatives': ['open setting', 'open set things', 'open set tings', 'open seting'],
        'intent': {'action': 'open_app', 'target': 'settings'},
    },
    {
        'phrase': 'open twitter',
        'model_name': 'open_twitter',
        'negatives': ['open twit ter', 'open twitch er', 'open twiter', 'open twitch'],
        'intent': {'action': 'open_app', 'target': 'twitter'},
    },
    {
        'phrase': 'open reddit',
        'model_name': 'open_reddit',
        'negatives': ['open red it', 'open read it', 'open red dit', 'open re dit'],
        'intent': {'action': 'open_app', 'target': 'reddit'},
    },
    {
        'phrase': 'open whatsapp',
        'model_name': 'open_whatsapp',
        'negatives': ['open whats app', 'open watts app', 'open what sap', 'open wats app'],
        'intent': {'action': 'open_app', 'target': 'whatsapp'},
    },
    {
        'phrase': 'open netflix',
        'model_name': 'open_netflix',
        'negatives': ['open net flix', 'open net flex', 'open net flicks', 'open met flix'],
        'intent': {'action': 'open_app', 'target': 'netflix'},
    },
    {
        'phrase': 'open claude',
        'model_name': 'open_claude',
        'negatives': ['open clod', 'open clawed', 'open clod e', 'open claud'],
        'intent': {'action': 'open_app', 'target': 'claude'},
    },
    {
        'phrase': 'open chatgpt',
        'model_name': 'open_chatgpt',
        'negatives': ['open chat gpt', 'open chat g p t', 'open chat jpt', 'open chad gpt'],
        'intent': {'action': 'open_app', 'target': 'chatgpt'},
    },
    {
        'phrase': 'open steam',
        'model_name': 'open_steam',
        'negatives': ['open steem', 'open stee m', 'open steam', 'open steme'],
        'intent': {'action': 'open_app', 'target': 'steam'},
    },
    {
        'phrase': 'open outlook',
        'model_name': 'open_outlook',
        'negatives': ['open out look', 'open look out', 'open outlok', 'open oot look'],
        'intent': {'action': 'open_app', 'target': 'outlook'},
    },
    {
        'phrase': 'mute volume',
        'model_name': 'mute_volume',
        'negatives': ['mute olume', 'moot volume', 'mute vol ume', 'nute volume'],
        'intent': {'action': 'volume_mute'},
    },
    {
        'phrase': 'take screenshot',
        'model_name': 'take_screenshot',
        'negatives': ['take screen shot', 'take screen shoot', 'take screen short', 'make screenshot'],
        'intent': {'action': 'screenshot'},
    },
    {
        'phrase': 'lock screen',
        'model_name': 'lock_screen',
        'negatives': ['lock screen', 'lock scream', 'look screen', 'log screen'],
        'intent': {'action': 'lock'},
    },
    {
        'phrase': 'new tab',
        'model_name': 'new_tab',
        'negatives': ['new tap', 'new tabb', 'nude tab', 'new tad'],
        'intent': {'action': 'browser_new_tab'},
    },
    {
        'phrase': 'close tab',
        'model_name': 'close_tab',
        'negatives': ['close tap', 'close tabb', 'clothes tab', 'close tad'],
        'intent': {'action': 'browser_close_tab'},
    },
    {
        'phrase': 'next tab',
        'model_name': 'next_tab',
        'negatives': ['next tap', 'text tab', 'next tabb', 'nest tab'],
        'intent': {'action': 'browser_next_tab'},
    },
    {
        'phrase': 'go back',
        'model_name': 'go_back',
        'negatives': ['go bat', 'go bak', 'go bacc', 'go beck'],
        'intent': {'action': 'browser_back'},
    },

    # ═══════════════════════════════════════════════════════════════════
    # CATEGORY B: Parameterized Commands (Type 2 — hybrid acoustic + STT)
    # The acoustic classifier detects the command PATTERN, then the
    # frontend records 3s of audio + runs STT to get the parameter.
    # ═══════════════════════════════════════════════════════════════════
    {
        'phrase': 'play song in spotify',
        'model_name': 'play_spotify',
        'negatives': ['open spotify', 'search on spotify', 'play on youtube', 'play song in youtube'],
        'intent': {'action': 'spotify_play', 'needs_param': True},
    },
    {
        'phrase': 'search on youtube',
        'model_name': 'search_youtube',
        'negatives': ['open youtube', 'search on google', 'search on github', 'play on youtube'],
        'intent': {'action': 'youtube_search', 'needs_param': True},
    },
    {
        'phrase': 'search on google',
        'model_name': 'search_google',
        'negatives': ['search on youtube', 'search on github', 'open google', 'search on reddit'],
        'intent': {'action': 'google_search', 'needs_param': True},
    },
    {
        'phrase': 'search on github',
        'model_name': 'search_github',
        'negatives': ['search on google', 'open github', 'search on youtube', 'search on gitlab'],
        'intent': {'action': 'github_search', 'needs_param': True},
    },
    {
        'phrase': 'play on youtube',
        'model_name': 'play_youtube',
        'negatives': ['play on spotify', 'search on youtube', 'open youtube', 'play song in spotify'],
        'intent': {'action': 'youtube_play', 'needs_param': True},
    },
    {
        'phrase': 'send message to',
        'model_name': 'send_message',
        'negatives': ['send massage to', 'send message', 'send mess age to', 'end message to'],
        'intent': {'action': 'send_message', 'needs_param': True},
    },
    {
        'phrase': 'set timer for',
        'model_name': 'set_timer',
        'negatives': ['set time for', 'set timmer for', 'set tie mer for', 'set timer four'],
        'intent': {'action': 'set_timer', 'needs_param': True},
    },
    {
        'phrase': 'set alarm for',
        'model_name': 'set_alarm',
        'negatives': ['set alarm four', 'set a larm for', 'set our alarm for', 'set al arm for'],
        'intent': {'action': 'set_alarm', 'needs_param': True},
    },
    {
        'phrase': 'create event',
        'model_name': 'create_event',
        'negatives': ['create vent', 'create a vent', 'create event', 'creat event'],
        'intent': {'action': 'create_event', 'needs_param': True},
    },
]

# ─── Summary ─────────────────────────────────────────────────────────
fixed_count = sum(1 for c in COMMANDS if not c['intent'].get('needs_param'))
param_count = sum(1 for c in COMMANDS if c['intent'].get('needs_param'))
print(f'Commands to train: {len(COMMANDS)}')
print(f'  Fixed (Type 1 — no STT):       {fixed_count}')
print(f'  Parameterized (Type 2 — STT):  {param_count}')
print()
for c in COMMANDS:
    tag = ' [PARAM]' if c['intent'].get('needs_param') else ''
    print(f'  {c["model_name"]:25s} ← "{c["phrase"]}"  ({len(c["negatives"])} negatives){tag}')
print(f'\nEstimated time: {len(COMMANDS) * 20} min ({len(COMMANDS) * 20 / 60:.1f} hrs)')
print(f'Disk per command: ~2GB clips (cleaned up after featurization)')
print(f'Output: {len(COMMANDS)} × ~800KB = ~{len(COMMANDS) * 0.8:.0f} MB total')

## 6. Training loop — trains each command sequentially

For each command:
1. Generate Piper TTS clips (positive = command phrase, negative = adversarial words)
2. Resample 22050 → 16000 Hz
3. Augment + extract features
4. Train DNN classifier (3-stage curriculum)
5. Ensemble + export ONNX
6. Download the `.onnx` file

The shared data (FMA, ACAV, RIRs) is loaded once and reused across all commands.

In [ ]:
import os, sys, copy, math, time, yaml, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.signal import resample_poly
from tqdm.auto import tqdm
import soundfile as sf
import glob

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ─── Check what data is available ─────────────────────────────────────
ACAV_TRAIN_PATH = '/content/acav_train_subset.npy'
ACAV_VAL_PATH = '/content/acav_val_subset.npy'
FMA_WAV_DIR = '/content/fma_wav'
RIR_DIR = '/content/mit_rirs'

ACAV_AVAILABLE = os.path.exists(ACAV_TRAIN_PATH) and os.path.exists(ACAV_VAL_PATH)
FMA_COUNT = len(os.listdir(FMA_WAV_DIR)) if os.path.exists(FMA_WAV_DIR) else 0
RIR_COUNT = len(os.listdir(RIR_DIR)) if os.path.exists(RIR_DIR) else 0

print(f'  ACAV100M: {"available" if ACAV_AVAILABLE else "NOT available — using synthetic negatives"}')
print(f'  FMA WAVs: {FMA_COUNT} files')
print(f'  MIT RIRs: {RIR_COUNT} files')

# ─── Google Drive checkpointing setup ─────────────────────────────────
# Defined in cell 4, but re-check here in case this cell is run independently
try:
    DRIVE_MOUNTED
except NameError:
    DRIVE_MOUNTED = False
try:
    DRIVE_MODELS_DIR
except NameError:
    DRIVE_MODELS_DIR = '/content/drive/MyDrive/nexus_command_models'

def save_to_drive(onnx_path, model_name):
    """Copy a trained model to Google Drive for persistence."""
    if not DRIVE_MOUNTED:
        return False
    try:
        drive_path = f'{DRIVE_MODELS_DIR}/{model_name}.onnx'
        shutil.copy2(onnx_path, drive_path)
        print(f'  saved to Drive: {drive_path} ({os.path.getsize(drive_path)/1e3:.0f} KB)')
        return True
    except Exception as e:
        print(f'  WARNING: Could not save to Drive: {e}')
        return False

def check_drive_for_model(model_name):
    """Check if a model already exists in Google Drive (resume capability)."""
    if not DRIVE_MOUNTED:
        return None
    drive_path = f'{DRIVE_MODELS_DIR}/{model_name}.onnx'
    if os.path.exists(drive_path) and os.path.getsize(drive_path) > 1000:
        return drive_path
    return None

# ─── Load ACAV if available, else generate synthetic negatives ────────
if ACAV_AVAILABLE:
    acav_train_np = np.load(ACAV_TRAIN_PATH, mmap_mode='r')
    acav_val_np = np.load(ACAV_VAL_PATH)
    M = acav_val_np.shape[0]
    val_listen_hours = M * 0.08 / 3600.0
    n_win_val = M - 16
    acav_val_windows = np.lib.stride_tricks.sliding_window_view(acav_val_np, (16, 96))[:, 0, :, :]
    acav_val_windows = np.ascontiguousarray(acav_val_windows.astype(np.float32))
    print(f'  ACAV train (mmap): {acav_train_np.shape}')
    print(f'  ACAV val windows: {acav_val_windows.shape} ({acav_val_windows.nbytes/1e6:.0f} MB, {val_listen_hours:.2f} hr)')
else:
    # Synthetic negatives: generate random feature-like data
    # This is a fallback — not as good as real ACAV100M, but allows training
    print('  Generating synthetic negative features (fallback)...')
    # Create ~100K random 16x96 windows as fake "ACAV" data
    SYNTH_N = 100_000
    acav_train_np = np.random.randn(SYNTH_N, 16, 96).astype(np.float32) * 0.5
    # Create a smaller val set (~10K windows)
    acav_val_np = np.random.randn(10_000, 16, 96).astype(np.float32) * 0.5
    M = acav_val_np.shape[0]
    val_listen_hours = M * 0.08 / 3600.0
    n_win_val = M - 16
    acav_val_windows = np.ascontiguousarray(acav_val_np.astype(np.float32))
    print(f'  Synthetic train: {acav_train_np.shape}')
    print(f'  Synthetic val: {acav_val_windows.shape} ({acav_val_windows.nbytes/1e6:.0f} MB)')
    print('  WARNING: Models trained with synthetic negatives will have higher false positives.')
    print('           Retrain with real ACAV100M when possible for production use.')
VAL_BATCH = 4096

# ─── DNN model (same architecture as wake-word) ──────────────────────
class WakewordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(16 * 96, 128)
        self.layernorm1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 1)
    def forward(self, x):
        return self.layer2(self.relu1(self.layernorm1(self.layer1(self.flatten(x)))))

# ─── Helper: generate clips for one command ──────────────────────────
def generate_clips_for_command(cmd, n_samples=2000, n_samples_val=1000):
    """Generate Piper TTS clips for a command phrase.

    Builds a config YAML that openWakeWord's train.py requires.
    ALL required keys are always present — empty lists when data
    is unavailable (train.py handles empty lists gracefully).
    """
    model_name = cmd['model_name']
    phrase = cmd['phrase']
    negatives = cmd['negatives']
    output_dir = f'/content/{model_name}_output'

    # ─── Build config with ALL required keys ──────────────────────────
    # train.py accesses these keys unconditionally (no .get() with defaults).
    # If any key is missing, train.py crashes with KeyError.
    # We always include every key, using empty lists when data is unavailable.

    config = {
        # --- Clip generation ---
        'target_phrase': [phrase],
        'model_name': model_name,
        'custom_negative_phrases': negatives,
        'n_samples': n_samples,
        'n_samples_val': n_samples_val,
        'tts_batch_size': 50,
        'piper_sample_generator_path': '/content/piper-sample-generator',

        # --- Augmentation ---
        'augmentation_rounds': 1,
        'augmentation_batch_size': 16,
        # 'total_length' is computed by train.py from generated clips — not set here

        # --- Background noise (FMA) ---
        # train.py line 664: always accesses both keys.
        # Empty lists are fine — augment_clips() just skips background noise.
        'background_paths': [FMA_WAV_DIR] if FMA_COUNT > 100 else [],
        'background_paths_duplication_rate': [1] if FMA_COUNT > 100 else [],

        # --- Room impulse responses ---
        # train.py line 662: always accesses this key.
        # Empty list is fine — augment_clips() just skips RIR convolution.
        'rir_paths': [RIR_DIR] if RIR_COUNT > 0 else [],

        # --- Output ---
        'output_dir': output_dir,
        'onnx_export': True,
        'tflite_export': False,

        # --- Training (used by our custom training code, not train.py --train_model) ---
        'steps': 20000,
        'max_negative_weight': 1500,
        'target_accuracy': 0.7,
        'target_recall': 0.5,
        'target_false_positives_per_hour': 0.5,
        'batch_size': 128,
        'learning_rate': 1e-4,
        'model_type': 'dnn',
        'layer_dim': 128,
        'layer_size': 128,
        'n_blocks': 1,
        'model_input_shape': [16, 96],
        'n_classes': 1,
        'batch_n_per_class': {
            'adversarial_negative': 50,
            'positive': 50,
        },

        # --- ACAV features (for our custom training code) ---
        # train.py --train_model needs these, but we use our own training code.
        # We still include them so the config is complete if someone uses --train_model.
        'feature_data_files': {},
        'false_positive_validation_data_path': '',
    }

    # Add ACAV features if available
    if ACAV_AVAILABLE:
        config['batch_n_per_class']['ACAV100M_sample'] = 1024
        config['feature_data_files'] = {'ACAV100M_sample': ACAV_TRAIN_PATH}
        config['false_positive_validation_data_path'] = ACAV_VAL_PATH

    os.makedirs(output_dir, exist_ok=True)
    config_path = f'/content/{model_name}_config.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)
    return config, config_path

# ─── Helper: resample clips ──────────────────────────────────────────
def resample_clips(output_dir):
    TARGET_SR = 16000
    wav_dirs = sorted({os.path.dirname(f)
                       for f in glob.glob(f'{output_dir}/**/*.wav', recursive=True)})
    for d in wav_dirs:
        files = [f for f in os.listdir(d) if f.endswith('.wav')]
        if not files: continue
        sr = sf.info(f'{d}/{files[0]}').samplerate
        if sr == TARGET_SR: continue
        for f in files:
            p = f'{d}/{f}'
            data, sr = sf.read(p)
            if sr != TARGET_SR:
                new_data = resample_poly(data.astype('float32'), TARGET_SR, sr)
                sf.write(p, new_data, TARGET_SR)
    # Clear stale features
    for f in glob.glob(f'{output_dir}/**/*.npy', recursive=True):
        os.remove(f)

# ─── Helper: run train.py with error capture ─────────────────────────
def run_train_py(config_path, mode, timeout=600):
    """Run openWakeWord train.py with full error output.

    Returns (success, output_string).
    Captures ALL output (not just tail -5) so we can diagnose failures.
    """
    cmd = f'{sys.executable} /content/openwakeword/openwakeword/train.py --training_config {config_path} --{mode}'
    print(f'  running: {mode}...')
    try:
        result = os.popen(cmd + ' 2>&1').read()
        # Print last 10 lines for visibility, but keep full output
        lines = result.strip().split('\n')
        for line in lines[-10:]:
            print(f'    {line}')
        if 'Traceback' in result or 'Error' in result:
            return False, result
        return True, result
    except Exception as e:
        return False, str(e)

# ─── Helper: train one command ───────────────────────────────────────
def train_command(cmd, config, config_path):
    model_name = cmd['model_name']
    FEAT = config['feature_save_dir'] if 'feature_save_dir' in config else f"{config['output_dir']}/{model_name}"

    # Check if already trained (local)
    onnx_path = f'{FEAT}/{model_name}.onnx'
    if os.path.exists(onnx_path):
        print(f'  SKIP {model_name}: already trained locally ({onnx_path})')
        return onnx_path

    # Check if already trained (Google Drive — resume from previous session)
    drive_path = check_drive_for_model(model_name)
    if drive_path:
        print(f'  SKIP {model_name}: found in Google Drive ({drive_path})')
        # Copy from Drive to local so downstream cells can find it
        os.makedirs(FEAT, exist_ok=True)
        shutil.copy2(drive_path, onnx_path)
        print(f'  copied from Drive to local: {onnx_path}')
        return onnx_path

    # ─── Step 1: Generate clips ───────────────────────────────────────
    dirs = {
        'positive_train': (f"{config['output_dir']}/{model_name}/positive_train", int(config['n_samples'] * 0.75)),
        'positive_test':  (f"{config['output_dir']}/{model_name}/positive_test", int(config['n_samples_val'] * 0.75)),
        'negative_train': (f"{config['output_dir']}/{model_name}/negative_train", int(config['n_samples'] * 0.75)),
        'negative_test':  (f"{config['output_dir']}/{model_name}/negative_test", int(config['n_samples_val'] * 0.75)),
    }
    all_full = all(os.path.isdir(p) and len(os.listdir(p)) >= exp for _, (p, exp) in dirs.items())
    if not all_full:
        print(f'  generating clips for "{cmd["phrase"]}"...')
        ok, output = run_train_py(config_path, 'generate_clips')
        if not ok:
            print(f'  ERROR: clip generation failed!')
            print(f'  Full output:\n{output}')
            raise RuntimeError(f'clip generation failed for {model_name}')

    # ─── Step 2: Resample 22050 → 16000 Hz ────────────────────────────
    print(f'  resampling clips to 16kHz...')
    resample_clips(config['output_dir'])

    # ─── Step 3: Augment + featurize ──────────────────────────────────
    needed = ['positive_features_train.npy', 'negative_features_train.npy',
              'positive_features_test.npy', 'negative_features_test.npy']
    if not all(os.path.exists(f'{FEAT}/{n}') for n in needed):
        print(f'  augmenting + featurizing...')
        ok, output = run_train_py(config_path, 'augment_clips')
        if not ok:
            print(f'  ERROR: augmentation failed!')
            print(f'  Full output:\n{output}')
            raise RuntimeError(f'augmentation failed for {model_name}')

    # Verify feature files exist
    for n in needed:
        path = f'{FEAT}/{n}'
        if not os.path.exists(path):
            raise RuntimeError(f'feature file missing: {path}')
        sz = os.path.getsize(path)
        if sz < 1000:
            raise RuntimeError(f'feature file too small ({sz} bytes): {path}')
        print(f'    {n}: {sz/1e6:.1f} MB')

    # ─── Step 4: Load features ────────────────────────────────────────
    pos_train = torch.from_numpy(np.load(f'{FEAT}/positive_features_train.npy').astype(np.float32)).to(DEVICE)
    neg_train = torch.from_numpy(np.load(f'{FEAT}/negative_features_train.npy').astype(np.float32)).to(DEVICE)
    pos_test  = torch.from_numpy(np.load(f'{FEAT}/positive_features_test.npy').astype(np.float32)).to(DEVICE)
    neg_test  = torch.from_numpy(np.load(f'{FEAT}/negative_features_test.npy').astype(np.float32)).to(DEVICE)
    print(f'  features: pos_train={tuple(pos_train.shape)} neg_train={tuple(neg_train.shape)}')

    # ─── Step 5: Train ────────────────────────────────────────────────
    model = WakewordModel().to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(reduction='none')
    TOTAL_STEPS = config['steps']
    MAX_NEG_W = config['max_negative_weight']
    TARGET_FP = config['target_false_positives_per_hour']
    THRESH = 0.5
    history = {'val_recall': [], 'val_accuracy': [], 'val_fp_per_hour': [], 'val_n_fp': [], 'loss': []}
    best_models = []

    B_POS, B_ANEG, B_ACAV = 32, 32, 64
    def random_acav_window_batch(k):
        N, T, F = acav_train_np.shape
        rows = np.random.randint(0, N, size=k)
        if T == 16:
            # Synthetic data is already (N, 16, 96) — no windowing needed
            return acav_train_np[rows].astype(np.float32)
        starts = np.random.randint(0, T - 16 + 1, size=k)
        out = np.empty((k, 16, F), dtype=np.float32)
        for i, (r, s) in enumerate(zip(rows, starts)):
            out[i] = acav_train_np[r, s:s+16, :].astype(np.float32)
        return out

    def build_batch():
        p_idx = torch.randint(0, pos_train.shape[0], (B_POS,), device=DEVICE)
        aneg_idx = torch.randint(0, neg_train.shape[0], (B_ANEG,), device=DEVICE)
        p, an = pos_train[p_idx], neg_train[aneg_idx]
        acav = torch.from_numpy(random_acav_window_batch(B_ACAV)).to(DEVICE)
        x = torch.cat([p, an, acav], dim=0)
        y = torch.cat([torch.ones(B_POS, device=DEVICE),
                       torch.zeros(B_ANEG + B_ACAV, device=DEVICE)])
        return x, y

    @torch.no_grad()
    def validate(label):
        model.eval()
        p_preds = torch.sigmoid(model(pos_test)).squeeze(-1)
        n_preds = torch.sigmoid(model(neg_test)).squeeze(-1)
        recall = (p_preds >= THRESH).float().mean().item()
        accuracy = (((p_preds >= THRESH).sum() + (n_preds < THRESH).sum()).item()
                    / (pos_test.shape[0] + neg_test.shape[0]))
        n_fp = 0
        for i in range(0, n_win_val, VAL_BATCH):
            chunk = torch.from_numpy(acav_val_windows[i:i+VAL_BATCH]).to(DEVICE)
            n_fp += (torch.sigmoid(model(chunk)).squeeze(-1) >= THRESH).sum().item()
        fp_per_hour = n_fp / max(val_listen_hours, 1e-6)
        history['val_recall'].append(recall)
        history['val_accuracy'].append(accuracy)
        history['val_fp_per_hour'].append(fp_per_hour)
        history['val_n_fp'].append(n_fp)
        save = False
        if len(history['val_n_fp']) >= 3:
            fp_p50 = np.percentile(history['val_n_fp'], 50)
            rc_p5 = np.percentile(history['val_recall'], 5)
            if n_fp <= fp_p50 and recall >= rc_p5:
                best_models.append((copy.deepcopy(model.state_dict()),
                                    {'val_recall': recall, 'val_accuracy': accuracy,
                                     'val_fp_per_hour': fp_per_hour, 'val_n_fp': n_fp}))
                save = True
        print(f'    [{label}] recall={recall:.3f} acc={accuracy:.3f} fp/hr={fp_per_hour:.2f} {"+" if save else "-"}', flush=True)
        model.train()

    def run_stage(idx, n_steps, lr, max_neg_w, val_window_frac=1.0):
        optimizer = optim.Adam(model.parameters(), lr=lr)
        weight_schedule = np.linspace(1.0, max_neg_w, n_steps)
        val_start = int(n_steps * (1.0 - val_window_frac))
        val_steps = set(np.linspace(val_start, n_steps - 1, 20).astype(int))
        warmup = max(1, n_steps // 5)
        hold = n_steps // 3
        accumulated = []
        t0 = time.time()
        for step in range(n_steps):
            # Heartbeat to prevent Colab idle timeout
            heartbeat(f'training {model_name} stage {idx} step {step}/{n_steps}')
            if step < warmup: lr_now = lr * (step + 1) / warmup
            elif step < warmup + hold: lr_now = lr
            else:
                decay_t = (step - warmup - hold) / max(1, n_steps - warmup - hold)
                lr_now = lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_t)))
            for pg in optimizer.param_groups: pg['lr'] = lr_now
            x, y = build_batch()
            logits = model(x).squeeze(-1)
            preds = torch.sigmoid(logits)
            keep = ((y == 0) & (preds >= 0.001)) | ((y == 1) & (preds < 0.999))
            if keep.sum() == 0:
                if step in val_steps: validate(f's{idx} {step}/{n_steps}')
                continue
            kept_logits = logits[keep]
            kept_y = y[keep]
            neg_w = weight_schedule[step]
            w = torch.where(kept_y > 0.5,
                            torch.tensor(1.0, device=DEVICE),
                            torch.tensor(neg_w, device=DEVICE, dtype=torch.float32))
            accumulated.append((kept_logits, kept_y, w))
            if sum(t[0].shape[0] for t in accumulated) >= 128:
                cat_logits = torch.cat([t[0] for t in accumulated])
                cat_y = torch.cat([t[1] for t in accumulated])
                cat_w = torch.cat([t[2] for t in accumulated])
                loss = (loss_fn(cat_logits, cat_y) * cat_w).mean()
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                history['loss'].append(loss.item())
                accumulated.clear()
            if step in val_steps:
                validate(f's{idx} {step}/{n_steps} ({(time.time()-t0)/60:.1f}m)')

    max_neg_w_now = MAX_NEG_W
    run_stage(1, TOTAL_STEPS, lr=1e-4, max_neg_w=max_neg_w_now, val_window_frac=0.25)
    if history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP:
        max_neg_w_now *= 2
    run_stage(2, max(2000, TOTAL_STEPS // 10), lr=1e-5, max_neg_w=max_neg_w_now)
    if history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP:
        max_neg_w_now *= 2
    run_stage(3, max(2000, TOTAL_STEPS // 10), lr=1e-6, max_neg_w=max_neg_w_now)
    print(f'  training done: {len(best_models)} checkpoints, best fp/hr={min(history["val_fp_per_hour"]):.2f}')

    # ─── Step 6: Ensemble + export ────────────────────────────────────
    if not best_models:
        final_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        accs = [s['val_accuracy'] for _, s in best_models]
        rcs = [s['val_recall'] for _, s in best_models]
        fps = [s['val_fp_per_hour'] for _, s in best_models]
        acc_p90 = np.percentile(accs, 90)
        rc_p90 = np.percentile(rcs, 90)
        fp_p10 = np.percentile(fps, 10)
        qualified = [(sd, sc) for sd, sc in best_models
                     if sc['val_accuracy'] >= acc_p90 and sc['val_recall'] >= rc_p90
                     and sc['val_fp_per_hour'] <= fp_p10]
        if not qualified:
            qualified = [sorted(best_models, key=lambda t: (t[1]['val_fp_per_hour'], -t[1]['val_recall']))[0]]
        keys = qualified[0][0].keys()
        final_state = {k: torch.stack([sd[k].float() for sd, _ in qualified]).mean(dim=0) for k in keys}
    model.load_state_dict(final_state)
    model.eval()

    class WakewordExportable(nn.Module):
        def __init__(self, base): super().__init__(); self.base = base
        def forward(self, x): return torch.sigmoid(self.base(x))
    export_model = WakewordExportable(model).to(DEVICE).eval()
    dummy = torch.randn(1, 16, 96, device=DEVICE)
    torch.onnx.export(export_model, dummy, onnx_path,
                      input_names=['onnx::Flatten_0'], output_names=['output'],
                      dynamic_axes={'onnx::Flatten_0': {0: 'batch'}, 'output': {0: 'batch'}},
                      opset_version=14, dynamo=False)
    print(f'  exported: {onnx_path} ({os.path.getsize(onnx_path)/1e3:.0f} KB)')

    # ─── Step 7: Sanity check ─────────────────────────────────────────
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path)
    with torch.no_grad():
        p_scores = sess.run(None, {sess.get_inputs()[0].name: pos_test.cpu().numpy()})[0].flatten()
        print(f'  sanity: recall@0.5={(p_scores >= 0.5).mean():.3f}')

    # ─── Step 8: Save to Google Drive ─────────────────────────────────
    save_to_drive(onnx_path, model_name)

    # ─── Step 9: Clean up clips to save disk ──────────────────────────
    # Each command generates ~2GB of clips. With 39 commands that's 78GB.
    # Delete clips after features are extracted to avoid running out of disk.
    try:
        shutil.rmtree(f"{config['output_dir']}/{model_name}/positive_train", ignore_errors=True)
        shutil.rmtree(f"{config['output_dir']}/{model_name}/positive_test", ignore_errors=True)
        shutil.rmtree(f"{config['output_dir']}/{model_name}/negative_train", ignore_errors=True)
        shutil.rmtree(f"{config['output_dir']}/{model_name}/negative_test", ignore_errors=True)
        print(f'  cleaned up clips (saved ~2GB disk)')
    except Exception:
        pass  # Non-critical

    return onnx_path

# ─── MAIN LOOP: train all commands ───────────────────────────────────
trained_models = []
failed_commands = []
for i, cmd in enumerate(COMMANDS):
    print(f'\n{"="*70}')
    print(f'Command {i+1}/{len(COMMANDS)}: "{cmd["phrase"]}" → {cmd["model_name"]}')
    print(f'{"="*70}')
    heartbeat(f'starting command {i+1}/{len(COMMANDS)}: {cmd["model_name"]}')
    try:
        config, config_path = generate_clips_for_command(cmd)
        onnx_path = train_command(cmd, config, config_path)
        trained_models.append((cmd, onnx_path))

        # Also try browser download (works if tab is focused)
        try:
            from google.colab import files
            files.download(onnx_path)
            print(f'  browser download triggered: {os.path.basename(onnx_path)}')
        except Exception as e:
            print(f'  browser download skipped: {e}')
            if DRIVE_MOUNTED:
                print(f'  model is safe in Google Drive: {DRIVE_MODELS_DIR}/{cmd["model_name"]}.onnx')
            else:
                print(f'  model saved at: {onnx_path} (ephemeral — mount Drive to persist!)')
    except Exception as e:
        print(f'  FAILED: {e}')
        failed_commands.append((cmd, str(e)))
        continue  # Don't abort the whole run — try the next command

print(f'\n{"="*70}')
print(f'DONE. Trained {len(trained_models)}/{len(COMMANDS)} command models.')
if failed_commands:
    print(f'FAILED: {len(failed_commands)} commands:')
    for cmd, err in failed_commands:
        print(f'  {cmd["model_name"]}: {err}')
print(f'{"="*70}')
for cmd, path in trained_models:
    print(f'  {cmd["model_name"]:20s} → {os.path.basename(path)} ({os.path.getsize(path)/1e3:.0f} KB)')
print(f'\nPlace all .onnx files at:')
print(f'  src-tauri/resources/oww/commands/')
if DRIVE_MOUNTED:
    print(f'\nAll models are also saved in Google Drive:')
    print(f'  {DRIVE_MODELS_DIR}/')

## 7. Export intent mapping JSON

This creates a `command_intents.json` file that NEXUS loads at startup to map each command model to its intent.

In [ ]:
import json, os, shutil

# Build intent map from ALL commands (not just trained ones).
# This ensures the JSON is always complete, even if some commands failed.
# NEXUS will just skip models that aren't present on disk.
intent_map = {}
for cmd in COMMANDS:
    intent_map[cmd['model_name']] = {
        'phrase': cmd['phrase'],
        'model_file': f'{cmd["model_name"]}.onnx',
        'intent': cmd['intent'],
    }

json_path = '/content/command_intents.json'
with open(json_path, 'w') as f:
    json.dump(intent_map, f, indent=2)
print(f'Wrote {json_path} ({len(intent_map)} commands):')
print(json.dumps(intent_map, indent=2))

# Save to Google Drive
if DRIVE_MOUNTED:
    drive_json_path = f'{DRIVE_MODELS_DIR}/command_intents.json'
    shutil.copy2(json_path, drive_json_path)
    print(f'\nSaved to Google Drive: {drive_json_path}')

# Browser download
try:
    from google.colab import files
    files.download(json_path)
    print(f'\nBrowser download triggered: command_intents.json')
except Exception as e:
    print(f'Browser download skipped: {e}')
    if DRIVE_MOUNTED:
        print(f'JSON is safe in Google Drive: {DRIVE_MODELS_DIR}/command_intents.json')

print(f'\nPlace at: src-tauri/resources/oww/commands/command_intents.json')

## After Training

All `.onnx` files and `command_intents.json` are saved in **two places**:

1. **Google Drive** (persistent): `MyDrive/nexus_command_models/`
   - Survives session disconnects
   - Use this as your primary source

2. **Browser downloads** (if tab was focused): your Downloads folder
   - Each model triggers a download as it completes

### To get the models into NEXUS:

Copy all files from Google Drive to:
```
src-tauri/resources/oww/commands/
  ├── open_youtube.onnx       (Type 1: fixed)
  ├── open_gmail.onnx         (Type 1: fixed)
  ├── open_chrome.onnx        (Type 1: fixed)
  ├── ...29 more fixed...
  ├── play_spotify.onnx       (Type 2: parameterized)
  ├── search_youtube.onnx     (Type 2: parameterized)
  ├── ...6 more parameterized...
  └── command_intents.json
```

NEXUS will load all command models at startup alongside the wake-word model.

### Command Types

- **Type 1 (Fixed)**: When the model fires, NEXUS executes the mapped intent directly — **no STT needed**. ~200ms response.
- **Type 2 (Parameterized)**: When the model fires, NEXUS speaks "On it sir", records 3s of audio, runs STT to get the parameter (song name, search query), then executes. ~2-4s response.

### If the session disconnected mid-training:

1. Re-open the notebook in Colab
2. **Runtime → Run all**
3. The notebook will:
   - Re-mount Google Drive
   - Detect models that were already trained (in Drive)
   - Skip those and resume from where it left off
4. The `command_intents.json` is always generated from the full COMMANDS list, so it's complete even if some models failed.